# ACE-Step 1.5 — free song-generation server for BibleMusically

Runs the **open-source, MIT-licensed [ACE-Step 1.5](https://github.com/ace-step/ACE-Step-1.5)** music model on a **free Kaggle/Colab GPU** and exposes its REST API through a public **cloudflared** tunnel.

Copy the printed `https://xxxx.trycloudflare.com` URL into the desktop app: **Settings → Music engine → ACE-Step → ACE-Step server URL**, then set the engine to *ACE-Step*.

**Before you run:** enable the GPU accelerator — Kaggle: *Settings → Accelerator → GPU T4 x2*; Colab: *Runtime → Change runtime type → GPU*.

> Free tunnel URLs and Kaggle sessions are temporary (sessions stop after ~9–12h idle, and the tunnel URL changes every run). Re-run this notebook and re-paste the URL when it expires.

## 1. Install ACE-Step 1.5

In [ ]:
# ── Upstream revision: report, and pin if the app asked for one ──────────────
# Rewritten by the app at push time from this engine's "Upstream revision" setting. Empty means
# track whatever upstream has published, which is the default and what every run did before.
_UPSTREAM_PINS = {}

def _upstream(path, name):
    """Check out a pinned ref if there is one, then print the revision actually in use."""
    import subprocess as _sp
    _ref = _UPSTREAM_PINS.get(name)
    if _ref:
        try:
            _sp.run(['git', '-C', path, 'fetch', '--depth', '1', 'origin', _ref], check=True)
            _sp.run(['git', '-C', path, 'checkout', '--detach', 'FETCH_HEAD'], check=True)
            print(f'[upstream] {name} pinned to {_ref}', flush=True)
        except Exception as _ex:
            print(f'[upstream] {name} could not be pinned to {_ref} ({_ex}) - using default branch', flush=True)
    try:
        _sha = _sp.run(['git', '-C', path, 'rev-parse', '--short', 'HEAD'],
                       capture_output=True, text=True).stdout.strip()
        _date = _sp.run(['git', '-C', path, 'log', '-1', '--format=%cs'],
                        capture_output=True, text=True).stdout.strip()
        print(f'[upstream] {name} {_sha} {_date}', flush=True)
    except Exception as _ex:
        print(f'[upstream] {name} revision unknown ({_ex})', flush=True)

import subprocess, sys, os

if not os.path.isdir('ACE-Step-1.5'):
    # NOTE: enable Internet in Session options first, or this clone fails with
    # "Could not resolve host: github.com".
    subprocess.run(['git','clone','--depth','1','https://github.com/ace-step/ACE-Step-1.5.git'], check=True)
_upstream('ACE-Step-1.5', 'ace-step')

# ACE-Step's pyproject.toml lists `nano-vllm` as a plain dependency, but it's actually resolved
# via [tool.uv.sources] to a *local path* inside the repo (acestep/third_parts/nano-vllm) — a
# uv-specific mechanism plain pip has no idea about. `pip install -e .` treats "nano-vllm" as an
# ordinary PyPI package name, finds no matching distribution for this platform, and the whole
# install fails before ACE-Step itself is ever installed (this is why the notebook used to error
# out here). `uv` — the installer the upstream project's own README uses — resolves
# [tool.uv.sources] correctly, so use that instead of plain pip for this one install.
import json

# ── Keep Kaggle's CUDA torch, whatever this engine's install wants ────────────
# The failure this prevents looked exactly like an exhausted GPU quota and was not one: the run had
# a T4 the whole time, but resolving this engine's dependencies pulled a CPU-only torch wheel over
# Kaggle's CUDA build. `torch.cuda.is_available()` then returns False, the serve cell refuses to
# start (it gates on torch, not on the driver), and the app reported "no GPU" on an account with
# hours to spare.
#
# Probed in a SUBPROCESS, twice over on purpose:
#   * torch caches CUDA availability for the life of a process, so an in-process check after a
#     reinstall would report the old answer;
#   * a pip reinstall cannot replace a module this process has already imported — so nothing here
#     may `import torch` in the kernel, or the repair could not take effect for the serve cell.
def _torch_probe():
    code = ('import json,torch;'
            'print(json.dumps({"v": torch.__version__, "cuda": torch.cuda.is_available()}))')
    r = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
    try:
        return json.loads(r.stdout.strip().splitlines()[-1])
    except Exception:
        return {'v': None, 'cuda': False, 'err': (r.stderr or '')[-300:]}

_torch_before = _torch_probe()
print('[torch] before install:', _torch_before.get('v'), 'cuda=', _torch_before.get('cuda'), flush=True)


def _repair_torch_if_clobbered():
    """Put back the exact CUDA wheel Kaggle shipped, if the install replaced it with a CPU one.

    Deliberately narrow. It runs only when the machine HAS a GPU and torch cannot see it, which is
    precisely the state where the notebook would refuse to serve anyway — so the worst case is that
    a broken run stays broken, and the best case is that it works. The version is not guessed: it is
    the build string Kaggle had before this cell touched anything, and its '+cuNNN' suffix names the
    wheel index to take it from.
    """
    try:
        smi = subprocess.run(['nvidia-smi'], capture_output=True, timeout=120).returncode
    except Exception:
        smi = 1
    after = _torch_probe()
    if smi != 0 or after.get('cuda'):
        print('[torch] after install:', after.get('v'), 'cuda=', after.get('cuda'), flush=True)
        return after
    prev = _torch_before.get('v') or ''
    if '+cu' not in prev:
        print('[torch] torch cannot see the GPU and there is no CUDA build to restore '
              '(was: %r). Not guessing.' % (prev,), flush=True)
        return after
    tag = prev.split('+', 1)[1]
    print('[torch] GPU present but torch is %s — restoring %s from the %s wheel index…'
          % (after.get('v'), prev, tag), flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps',
                    'torch==' + prev, '--index-url', 'https://download.pytorch.org/whl/' + tag])
    after = _torch_probe()
    print('[torch] after repair:', after.get('v'), 'cuda=', after.get('cuda'), flush=True)
    return after

subprocess.run([sys.executable,'-m','pip','install','-q','uv'], check=True)
subprocess.run(['uv','pip','install','--system','-q','-e','./ACE-Step-1.5'], check=True)

_torch_after = _repair_torch_if_clobbered()


# ---- Verify the environment actually works (this, not pip's red text, is what matters) ----
try:
    import torch
    torch_ok = True
    print(f'  ✅ torch        {torch.__version__} (cuda={torch.cuda.is_available()})')
except Exception as ex:
    torch_ok = False
    print(f'  ❌ torch        FAILED: {ex}')
try:
    import acestep  # package name per ACE-Step-1.5 pyproject
    print('  ✅ acestep      importable')
except Exception as ex:
    print(f'  ⚠️  acestep import raised (the acestep-api CLI may still serve): {ex}')
print('\nACE-Step installed — environment OK.' if torch_ok
      else '\n⚠️  torch import FAILED — fix that before continuing.')


## 2. Install cloudflared (public tunnel, no signup)

In [ ]:
import subprocess, os

if subprocess.run(['which', 'cloudflared'], capture_output=True).returncode != 0:
    subprocess.run(
        'curl -L --output /tmp/cloudflared.deb '
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb '
        '&& dpkg -i /tmp/cloudflared.deb',
        shell=True, check=True)
print('cloudflared ready.')

## 3. (Optional) set an API key

Leave blank for the simplest setup. If you set one, paste the **same** value into the app's *API key* field so requests are accepted.

In [ ]:
import os

API_KEY = ''  # e.g. 'my-secret-key' — must match the app's ACE-Step API key field
if API_KEY:
    os.environ['ACESTEP_API_KEY'] = API_KEY
os.environ['ACESTEP_API_HOST'] = '127.0.0.1'
os.environ['ACESTEP_API_PORT'] = '8001'
print('API key set.' if API_KEY else 'No API key (open server).')

## 4. Launch the ACE-Step REST server + tunnel

Starts `acestep-api` on port 8001, opens a public cloudflared tunnel to it, and prints the URL to paste into the app. Keep this cell running — closing it stops the server.

In [ ]:
import subprocess, sys, time, re, os, threading

PORT = 8001

# Start the ACE-Step REST API server. First run downloads model weights (a few minutes).
api = subprocess.Popen(
    ['acestep-api'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ})

def _pump(proc, tag):
    for line in proc.stdout:
        print(f'[{tag}] {line}', end='')

threading.Thread(target=_pump, args=(api, 'acestep'), daemon=True).start()

# Wait for the server to accept connections.
import urllib.request, urllib.error, json
print('Waiting for ACE-Step server to come up (first run downloads weights)...')
for _ in range(120):
    try:
        req = urllib.request.Request(f'http://127.0.0.1:{PORT}/query_result',
                                     data=json.dumps({'task_id_list': []}).encode(),
                                     headers={'Content-Type': 'application/json'})
        urllib.request.urlopen(req, timeout=5)
        print('Server is up.')
        break
    except urllib.error.HTTPError:
        print('Server is up (HTTP response received).')
        break
    except Exception:
        time.sleep(5)
else:
    print('WARNING: server did not respond in time; check the [acestep] logs above.')

# Open the public tunnel and print its URL.

# ── Batch-run guard v2 ──────────────────────────────────────────────
# Source-update pushes run GPU-less and should exit fast; a GPU batch run is a
# DELIBERATE server start (the app's "Start server" button pushes with GPU on)
# and must open the tunnel and keep serving.
_is_batch = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive').lower() == 'batch'

# Two independent questions, asked separately because they fail apart — and which one failed IS
# the diagnosis:
#   nvidia-smi  — does this container have a GPU and a working driver at all?
#   torch.cuda  — can the framework that runs the model actually reach it?
# A GPU-off source-update push answers no to both, and so does Kaggle declining an accelerator.
# An install step that replaced Kaggle's CUDA torch with a CPU-only wheel answers YES to the
# first and no to the second. The old guard asked only nvidia-smi and reported every "no" as an
# exhausted weekly quota — which sent people to a quota page that had 29.8 of 30 hours left on it.
_smi_rc, _smi_note = None, ''
try:
    _smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=120)
    _smi_rc = _smi.returncode
    _smi_note = (_smi.stderr or '').strip().replace('\n', ' ')[:200]
except FileNotFoundError:
    _smi_note = 'nvidia-smi is not installed on this container'
except Exception as _ex:
    _smi_note = '{}: {}'.format(type(_ex).__name__, _ex)
try:
    import torch as _t
    _torch_cuda, _torch_ver = bool(_t.cuda.is_available()), _t.__version__
except Exception as _ex:
    _torch_cuda, _torch_ver = False, 'unavailable ({})'.format(type(_ex).__name__)

# Serving is gated on torch rather than on the driver, because the model is loaded onto whatever
# device torch reports. A container that has a GPU torch cannot see would otherwise open a public
# tunnel to a server generating on CPU — minutes of audio at hours of wall clock, which is a worse
# outcome than not starting, and much harder to diagnose from the app.
_has_gpu = _torch_cuda
if _is_batch and not _has_gpu:
    print('=' * 70)
    print('  NO GPU ON THIS RUN — not serving.')
    print('  nvidia-smi: ' + ('exit {}'.format(_smi_rc) if _smi_rc is not None else 'did not run')
          + (' — {}'.format(_smi_note) if _smi_note else ''))
    print('  torch {}: cuda.is_available() = {}'.format(_torch_ver, _torch_cuda))
    print('  CUDA_VISIBLE_DEVICES = {!r}'.format(os.environ.get('CUDA_VISIBLE_DEVICES', '<unset>')))
    if _smi_rc == 0:
        print('  GPU PRESENT BUT TORCH CANNOT USE IT — an install step in this notebook replaced')
        print("  Kaggle's CUDA build of torch with a CPU-only one. Fix that cell; the quota is")
        print('  not the problem here.')
    else:
        print('  KAGGLE GAVE THIS SESSION NO ACCELERATOR. The app always asks for one, so this is')
        print('  the scheduler declining: the weekly quota is spent, both GPU session slots are')
        print('  busy, or no GPU was free at that moment. That last case is common and transient —')
        print('  if the quota page still shows hours left, simply start again.')
        print('  Quota: https://www.kaggle.com/settings  (Accelerator usage).')
    print('  (A deliberate GPU-off push is just a cheap source update - nothing is wrong.)')
    print('=' * 70)
    print('Start the server from the app (Start server button) or run interactively with GPU on.')
else:
    import urllib.request, urllib.error

    # ── Reliable public tunnel with self-healing ───────────────────────────────
    # The failure this fixes: cloudflared prints a *.trycloudflare.com URL and even registers an
    # edge connection, yet the Cloudflare edge never actually ROUTES the hostname, so the URL never
    # answers and the app times out. Quick tunnels are flaky per-process and QUIC (UDP) egress can
    # be throttled. So we: (1) probe our OWN public URL to confirm it truly routes, (2) auto-restart
    # cloudflared — first over QUIC, then over HTTP/2, which survives UDP throttling — and (3) fall
    # back to localhost.run (ssh) if cloudflared keeps failing. Only a URL that actually ANSWERS is
    # printed as ready, and every step is logged so a failure is diagnosable from the app's log tail.
    _url_re = re.compile(r'https://[-a-z0-9]+\.(?:trycloudflare\.com|lhr\.life|serveo\.net)')

    def _probe_public(url, timeout=8):
        # True iff the tunnel truly routes: ANY HTTP status < 500 back proves the edge reached our
        # server. A connection error/timeout, or Cloudflare's own 5xx (e.g. 530 = tunnel down),
        # means "not routed yet".
        try:
            with urllib.request.urlopen(url.rstrip('/') + '/', timeout=timeout) as r:
                return r.status < 500
        except urllib.error.HTTPError as he:
            return he.code < 500
        except Exception:
            return False

    def _pump(proc, holder, tag='tunnel'):
        def _run():
            for line in proc.stdout:
                print(f'[{tag}] {line}', end='')
                m = _url_re.search(line)
                if m and not holder.get('url'):
                    holder['url'] = m.group(0)
        threading.Thread(target=_run, daemon=True).start()

    def _spawn_cloudflared(protocol):
        args = ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}']
        if protocol:
            args += ['--protocol', protocol]
        print(f'[tunnel] launching cloudflared (protocol={protocol or "auto"})...', flush=True)
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        h = {}; _pump(p, h); return p, h

    def _spawn_localhostrun():
        print('[tunnel] launching localhost.run over ssh...', flush=True)
        p = subprocess.Popen(
            ['ssh', '-o', 'StrictHostKeyChecking=no', '-o', 'UserKnownHostsFile=/dev/null',
             '-o', 'ServerAliveInterval=30', '-R', f'80:localhost:{PORT}', 'nokey@localhost.run'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        h = {}; _pump(p, h); return p, h

    def _bring_up(proc, holder, url_wait=30, route_wait=75):
        t0 = time.time()
        while time.time() - t0 < url_wait and not holder.get('url'):
            if proc.poll() is not None:
                print('[tunnel] process exited before printing a URL.', flush=True); return None
            time.sleep(1)
        url = holder.get('url')
        if not url:
            print(f'[tunnel] no URL within {url_wait}s.', flush=True); return None
        print(f'Tunnel URL: {url}', flush=True)
        print('Waiting for the edge to route it...', flush=True)
        t1 = time.time()
        while time.time() - t1 < route_wait:
            if proc.poll() is not None:
                print('[tunnel] tunnel process exited during the routing wait.', flush=True); return None
            if _probe_public(url):
                print(f'[tunnel] OK: answered from inside Kaggle after {int(time.time()-t1)}s (a brand-new address can still take minutes to route elsewhere).', flush=True)
                return url
            time.sleep(4)
        print(f'[tunnel] {url} never answered within {route_wait}s - treating as dead.', flush=True)
        return None

    _attempts = [('cf', 'quic'), ('cf', 'http2'), ('lhr', None)]
    active_proc = None; public_url = None
    for _i, (_kind, _proto) in enumerate(_attempts, 1):
        print(f'\n[tunnel] ===== attempt {_i}/{len(_attempts)}: {_kind} {_proto or ""} =====', flush=True)
        try:
            _p, _h = _spawn_cloudflared(_proto) if _kind == 'cf' else _spawn_localhostrun()
        except FileNotFoundError as _e:
            print(f'[tunnel] cannot launch ({_e}); skipping this attempt.', flush=True); continue
        _routed = _bring_up(_p, _h)
        if _routed:
            active_proc, public_url = _p, _routed; break
        try: _p.terminate()
        except Exception: pass
        time.sleep(2)

    print('\n' + '=' * 70)
    if public_url:
        print('  PASTE THIS INTO THE APP  ->  Settings -> ACE-Step server URL:')
        print(f'  {public_url}')
    else:
        print('  FAILED: no working public tunnel after all attempts. The local server is fine,')
        print('  but nothing outside can reach it - retry "Start & connect" from the app.')
    print('=' * 70, flush=True)

    if public_url and active_proc:
        # ── Idle-shutdown watchdog ──────────────────────────────────────
        # After IDLE_SHUTDOWN_MIN minutes with no ESTABLISHED connection to the server port, stop the
        # tunnel so a forgotten run stops burning GPU quota. App polling / liveness counts as activity.
        IDLE_SHUTDOWN_MIN = 30  # was 15: a quick tunnel can need >12 min to become routable, and
        # nothing connects to the port until it does — so the watchdog was ending healthy runs
        # before anyone could reach them. 30 still stops a forgotten session inside the hour.
        def _idle_watchdog():
            _port_hex = ':%04X' % PORT
            _last = time.time()
            while True:
                time.sleep(30)
                _active = False
                for _tbl in ('/proc/net/tcp', '/proc/net/tcp6'):
                    try:
                        with open(_tbl) as _f:
                            for _l in _f.readlines()[1:]:
                                _q = _l.split()
                                if _q[1].endswith(_port_hex) and _q[3] == '01':
                                    _active = True; break
                    except OSError:
                        _active = True
                    if _active: break
                if _active:
                    _last = time.time()
                elif time.time() - _last > IDLE_SHUTDOWN_MIN * 60:
                    print(f'[watchdog] No requests for {IDLE_SHUTDOWN_MIN} min - shutting down to save GPU quota.', flush=True)
                    try: active_proc.terminate()
                    except Exception: pass
                    return
        threading.Thread(target=_idle_watchdog, daemon=True).start()
        print(f'Keep this cell running. Idle watchdog armed: auto-stops after {IDLE_SHUTDOWN_MIN} min idle.', flush=True)
        active_proc.wait()